<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 6 - Manage Your Fund (DRAFT)

Your Studio 5 fund has been live on the class leaderboard for a few weeks. Tonight you are its manager.

1. **Part 1 - Load my fund.** Paste your record link.
2. **Part 2 - What happened?** The race so far, stock by stock.
3. **Part 3 - Look around.** What the class holds, and what is in the news.
4. **Part 4 - What does the model say today?** Solve Ms. Womack's model again with the newest data.
5. **Part 5 - My decision.** Hold or trade, with a reason, sent to the leaderboard.

Every cell is filled in: read it, run it, read what it prints. The notebook comes loaded with **Dr. Dave's example
fund**, so it runs from top to bottom as it is. Your job is to run it on **your** fund.

**After tonight.** You can come back and change your fund as often as you like until the market closes on
**Thursday, Dec 3**. The fund worth the most at that close wins the class race, and Dr. Dave writes its manager a
LinkedIn recommendation. This notebook is also your **final project**: the last section prints the numbers for your
2-page report, due **Monday, Dec 7**. Your grade does not depend on making money.

## Import modules

Same setup as Studio 5: Pyomo, the two solvers (ipopt and Bonmin) and yfinance. Run this first, every time. It takes
about a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import json
import requests
import yfinance as yf
from pyomo.environ import *
from IPython.display import display
import matplotlib.dates as mdates

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Part 1 - Load my fund

When you sent your fund in Studio 5, the leaderboard made a **private record** for it: one link that holds your
stocks, your rules and your fund's value at every market close.

**Where is my link?** In HuskyCT, open **Gradebook**, click **Studio 5 credit**, and read the feedback. Copy the whole
link (it ends in `.json`). Keep it to yourself.

**No fund on the board yet?** Open the Studio 5 notebook, Run all, and send your fund from the last cell. The receipt
prints your record link. Then come back here.

### Step 1 - Paste my record link

The link below is Dr. Dave's example fund. Replace it with yours.

In [ ]:
record_link = 'https://storage.googleapis.com/opim5641-leaderboard-f26/records/8000437f223b5f668e7dee73.json'

### Step 2 - Read my record

In [ ]:
# ask the leaderboard for my record
my_record = requests.get(record_link.strip(), timeout=30).json()
my_name = my_record['nickname']

print("Fund:    ", my_name)
print("About:   ", my_record['theme'])
print("Updated: ", my_record['updated'])

### Step 3 - The versions of my fund

Every time you sent your fund, the leaderboard saved a **version**. A version starts counting at the first market
close after you sent it. Most of you have one version. If you re-sent your fund, you have more.

In [ ]:
# one row per version that has started counting, oldest first
versions = []
for entry in my_record['entries']:
    if entry['counts_from'] is None:
        # sent, but still waiting for the next market close
        continue
    version = {
        'day': pd.Timestamp(entry['counts_from']),          # the close it counts from
        'reason': entry['reason'],                          # the reason I typed when I sent it
        'mix': entry['portfolios']['with_rules'],           # my fund: stock -> share of the money
        'twin_mix': entry['portfolios']['no_rules'],        # its no-rules twin
    }
    if len(versions) > 0 and versions[-1]['day'] == version['day']:
        # two versions sent before the same close: only the later one counts
        versions[-1] = version
    else:
        versions.append(version)

print(len(versions), "version(s) of my fund have started counting")

**Only if you sent your fund today.** A fund sent today has not started counting yet, so there is no race to look
at. The next cell sets up a **practice run** instead: what your fund would have done if you had bought it at the Nov 5
close, the day most of the class started. For everyone else this cell changes nothing.

In [ ]:
practice = len(versions) == 0
if practice:
    newest = my_record['entries'][-1]
    versions.append({
        'day': pd.Timestamp('2026-11-05'),
        'reason': 'practice run (my real fund starts counting at the next close)',
        'mix': newest['portfolios']['with_rules'],
        'twin_mix': newest['portfolios']['no_rules'],
    })
    print("Practice run: my fund as if I had bought it at the Nov 5 close.")

# show the versions
rows = []
for version in versions:
    rows.append({'counts from': version['day'].strftime('%a %b %d'), 'stocks held': len(version['mix']), 'my reason': version['reason']})
pd.DataFrame(rows)

### Step 4 - My stocks, my rules and my risk limit

These come from the newest version of your fund. They are the same `my_stocks`, `rules` and `extra_rules` dictionaries
you filled in for Studio 5, and the risk limit you sent.

In [ ]:
newest = my_record['entries'][-1]

record_stocks = newest['sectors']            # every stock on my list: ticker -> sector
record_rules = newest['rules']               # the floor, the cap, the sectors, the most stocks
record_extra = newest['extra_rules']         # the extra linking rules from the playground
record_risk_limit = newest['risk_limit']     # the risk limit I sent

print("My list:", len(record_stocks), "stocks")
print(", ".join(sorted(record_stocks)))
print()
print("My rules:      ", record_rules)
print("My extra rules:", record_extra)
print("My risk limit:  %.4f" % record_risk_limit)

The model did not buy every stock on your list. These are the ones it bought in your newest version, and the
share of your money it put in each.

In [ ]:
bought_as = versions[-1]['mix']

rows = []
for t in bought_as:
    rows.append({'stock': t, 'sector': record_stocks.get(t, ''), 'share I bought (%)': round(100 * bought_as[t], 1)})
pd.DataFrame(rows).set_index('stock').sort_values('share I bought (%)', ascending=False)

### Step 5 - The class crowd fund

The **class crowd fund** is the average of every student's Studio 5 fund: each student's fund counts once. It is a
benchmark made of your classmates. The file also lists every student fund by its leaderboard name, with the stocks it
holds.

In [ ]:
crowd_file = requests.get('https://storage.googleapis.com/opim5641-leaderboard-f26/crowd.json', timeout=30).json()

crowd_mix = crowd_file['crowd']['weights']     # the crowd fund: stock -> share
class_funds = crowd_file['named_funds']        # every student fund: its leaderboard name and its mix

print("The crowd fund is the average of", len(class_funds), "student funds:")
for fund in class_funds:
    print("   ", fund['name'], "-", len(fund['weights']), "stocks")
print("Together they hold", len(crowd_mix), "different stocks.")

#### How is the crowd fund worked out?

Put the same number of dollars in every student fund. Then a stock's share of the crowd fund is the **average of its
share across all of the funds**, counting 0% for a fund that does not hold it.

One stock by hand, the one the most funds hold:

In [ ]:
# the stock with the biggest share of the crowd fund
example = pd.Series(crowd_mix).idxmax()

# add up its share in every fund (0 if the fund does not hold it), then divide by the number of funds
total_share = 0
holding_it = 0
for fund in class_funds:
    if example in fund['weights']:
        total_share += fund['weights'][example]
        holding_it += 1

print(example, "is held by", holding_it, "of the", len(class_funds), "funds.")
print("Their shares of it add up to %.1f%%." % (100 * total_share))
print("Divide by %d funds: %.1f%% / %d = %.2f%%" % (len(class_funds), 100 * total_share, len(class_funds), 100 * total_share / len(class_funds)))
print("The crowd file says:  %.2f%%" % (100 * crowd_mix[example]))

The ten biggest holdings of the crowd fund. All of its shares add up to 100%.

In [ ]:
crowd_table = pd.Series(crowd_mix).sort_values(ascending=False) * 100

print("The crowd fund's shares add up to %.1f%%" % crowd_table.sum())
crowd_table.head(10).round(2).to_frame('share of the crowd fund (%)')

### Step 6 - Download the prices

We need daily closing prices for three groups of tickers. The steps below build the list, download it once from
Yahoo Finance, and tidy it up.

#### 6a - Which tickers do we need?

In [ ]:
# 1. every stock on my list (Part 4 solves the model on these; the model did not buy all of them)
mine = list(record_stocks)

# 2. any stock an older version of my fund held (Part 2 needs it)
older = []
for version in versions:
    older = older + list(version['mix']) + list(version['twin_mix'])

# 3. every stock the class holds (Part 3 looks at these)
class_stocks = list(crowd_mix)

# 4. the Studio 5 starter list (Part 4 uses it for an example about NVIDIA)
starter = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'GS', 'V', 'JNJ', 'LLY', 'UNH', 'XOM', 'CVX', 'KO', 'WMT',
           'COST', 'AMZN', 'HD', 'CAT', 'LMT']

# one list, each ticker once. It starts with SPY, a fund that tracks the S&P 500: our benchmark.
tickers = ['SPY']
print("SPY (the S&P 500):                      1 ticker")

groups = {'My list': mine, 'Older versions of my fund': older, 'The class': class_stocks, 'The Studio 5 starter list': starter}
for name in groups:
    added = 0
    for t in groups[name]:
        if t not in tickers:
            tickers.append(t)
            added += 1
    print("%-28s adds %3d new tickers (%d in the group, the rest are already on the list)" % (name + ':', added, len(set(groups[name]))))

print("To download:                         %3d tickers in all" % len(tickers))

#### 6b - Download them

Daily prices from September 2021 to today, adjusted for splits and dividends. About 20 seconds.

In [ ]:
try:
    prices = yf.download(tickers, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)['Close']
except Exception as problem:
    prices = pd.DataFrame()

if len(prices) > 0:
    print("Yahoo answered:", prices.shape[0], "days of prices for", prices.shape[1], "tickers.")
else:
    print("Yahoo did not answer. The next step uses the backup copy.")

#### 6c - The backup copy

Yahoo sometimes blocks Colab, or skips a ticker. Then we use a copy of the same prices saved for class. If Yahoo
answered in 6b, this step changes nothing, and it says so.

In [ ]:
class_copy = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)
used_backup = False

# Yahoo gave us nothing: use the class copy for everything
if 'SPY' not in prices.columns or prices['SPY'].notna().sum() < 200:
    prices = class_copy.copy()
    used_backup = True
    print("Yahoo did not answer, so ALL of the prices come from the class copy.")

# Yahoo skipped a ticker: take that one from the class copy
for t in tickers:
    if t not in prices.columns or prices[t].notna().sum() == 0:
        used_backup = True
        if t in class_copy.columns:
            prices[t] = class_copy[t].reindex(prices.index)
            print("Yahoo skipped", t, "- taken from the class copy.")
        else:
            # no prices anywhere: treat it as cash (a price that never moves)
            prices[t] = 1.0
            print("No prices anywhere for", t, "- treated as cash.")

if not used_backup:
    print("Yahoo gave us every ticker, so the backup copy was not needed.")

#### 6d - Keep closed days only

Today's price keeps moving until the market closes at 4 PM Eastern. So before the close we stop at yesterday.

In [ ]:
# put the days in order, and fill a missing day with the day before
prices = prices.sort_index().ffill()

# before 4:30 PM Eastern, drop today's row (it is not a closing price yet)
now = pd.Timestamp.now(tz='America/New_York')
if now.hour * 60 + now.minute < 16 * 60 + 30:
    prices = prices[prices.index < pd.Timestamp(now.date())]

last_close = prices.index[-1]
print("Prices through the close of", last_close.strftime('%a %b %d, %Y'))
print("The `prices` table has", prices.shape[1], "tickers. Below: the last three closes for only the", len(bought_as), "stocks I hold.")

prices[list(bought_as)].tail(3).round(2)

# Part 2 - What happened to my fund?

### 2a - The race so far

The leaderboard gave your fund \$10,000 at its first close and then **held** it: no trading, each stock's dollars
just grow or shrink with its price. That is the same `hold` idea as Studio 5, Part 4c.

We compare four lines, all starting with \$10,000 on your fund's first day:

* **my fund** (with my rules)
* **its no-rules twin** (same stocks, same risk limit, no business rules)
* **the class crowd fund**
* **the S&P 500**

#### My fund and its twin

The leaderboard already worked out what these two were worth at every close. We just read them from your record.

In [ ]:
if practice:
    # a practice run has no leaderboard values yet, so hold the mix from the practice day (the function is below)
    window = prices.loc[versions[0]['day']:]
    my_line = 0
    twin_line = 0
    for t in versions[0]['mix']:
        my_line = my_line + 10000 * versions[0]['mix'][t] * window[t] / window[t].iloc[0]
    for t in versions[0]['twin_mix']:
        twin_line = twin_line + 10000 * versions[0]['twin_mix'][t] * window[t] / window[t].iloc[0]
else:
    days = pd.to_datetime(my_record['values']['dates'])
    my_line = pd.Series(my_record['values']['with_rules'], index=days)      # my fund, in dollars, at every close
    twin_line = pd.Series(my_record['values']['no_rules'], index=days)      # its no-rules twin

first_day = my_line.index[0]
print("My fund's first day:", first_day.strftime('%a %b %d'))
print("Trading days since: ", len(my_line) - 1)

# the last three closes
my_line.tail(3).round(0).to_frame('my fund ($)')

#### The crowd fund and the S&P 500

These two are not in your record, so we hold them ourselves, from your first day, so the comparison is fair.

`hold` splits \$10,000 across a mix on one day and lets each stock's dollars follow its price:

$$\text{value on day } t = \sum_k \$10{,}000 \times x_k \times \frac{P_{k,t}}{P_{k,\text{first day}}}$$

In [ ]:
def hold(mix, day):
    # $10,000 split by `mix` at the close of `day`, then held. Returns the value at every close after that.
    window = prices.loc[day:]
    line = 0
    for t in mix:
        line = line + 10000 * mix[t] * window[t] / window[t].iloc[0]
    return line


crowd_line = hold(crowd_mix, first_day)
spy_line = hold({'SPY': 1.0}, first_day)

print("The crowd fund today: $%.0f" % crowd_line.iloc[-1])
print("The S&P 500 today:    $%.0f" % spy_line.iloc[-1])

#### The scorecard

Where each \$10,000 stands today. **Worst drop** is the biggest fall from a high point along the way.

In [ ]:
lines = {my_name: my_line, 'its no-rules twin': twin_line, 'class crowd fund': crowd_line, 'S&P 500': spy_line}

rows = {}
for name in lines:
    line = lines[name]
    row = {}
    row['$10,000 became'] = round(float(line.iloc[-1]))
    row['return (%)'] = round(100 * (line.iloc[-1] / line.iloc[0] - 1), 2)
    row['worst drop (%)'] = round(100 * (1 - line / line.cummax()).max(), 2)
    rows[name] = row
scorecard = pd.DataFrame(rows).T
scorecard

#### The picture

In [ ]:
fig, panel = subplots(figsize=(12, 5.5))
panel.plot(spy_line.index, spy_line.values, color='black', linewidth=2, label='S&P 500')
panel.plot(twin_line.index, twin_line.values, color='gray', linewidth=1.5, linestyle='--', label='no-rules twin')
panel.plot(crowd_line.index, crowd_line.values, color='tab:purple', linewidth=1.5, linestyle=':', label='class crowd fund')
panel.plot(my_line.index, my_line.values, color='tab:green', linewidth=3, marker='o', markersize=4, label=my_name)

# a dotted line on the day each later version of my fund started
for k in range(1, len(versions)):
    panel.axvline(versions[k]['day'], color='tab:green', linestyle=':', linewidth=1)

panel.axhline(10000, color='lightgray', linewidth=1)
panel.set_title('$10,000 since the %s close' % first_day.strftime('%b %d'))
panel.set_ylabel('dollars')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
panel.legend()
show()

**Read the race.** Is the green line above or below the black one? Above the gray one?

### 2b - Stock by stock

In Studio 5 we worked in **returns**, which are percentages. Tonight we turn each stock's return into **dollars**,
because dollars add up: the dollars from every stock sum to your fund's gain or loss.

$$\text{dollars a stock added} = \text{dollars I put in it} \times \text{its return}$$

A small share in a stock with a big return can matter more than a big share in a stock that did nothing.

#### Where this starts

We look at the stocks you hold **now**, from the day you bought them. For most of you that is your fund's first day.
If you re-sent your fund, it is the day your newest version started, and the money you started with is whatever the
fund was worth that day.

In [ ]:
bought_day = versions[-1]['day']                       # the close my newest version counts from
money_at_start = float(my_line.loc[bought_day])        # what my fund was worth at that close

print("I bought the stocks I hold now at the", bought_day.strftime('%a %b %d'), "close.")
print("My fund was worth $%.0f that day." % money_at_start)

#### What each stock is worth today

For each stock: the dollars you put in, times how much its price has moved since.

In [ ]:
dollars_added = {}       # stock -> the dollars it added (or took away)
worth_today = {}         # stock -> what my holding is worth today

for t in bought_as:
    bought = money_at_start * bought_as[t]                                  # dollars I put in this stock
    worth = bought * prices.loc[last_close, t] / prices.loc[bought_day, t]  # what they became
    worth_today[t] = worth
    dollars_added[t] = worth - bought

# add up what every stock is worth
my_worth = 0
for t in worth_today:
    my_worth += worth_today[t]

print("My fund today: $%.0f" % my_worth)
print("(The leaderboard says $%.0f. A few dollars apart is normal.)" % my_line.iloc[-1])

#### My shares today

**Share today** is the percent of your fund's dollars that sits in each stock right now. It is not the share you
bought: a stock that went up is now a bigger slice, and a stock that fell is a smaller one. The shares always add up
to 100%.

In [ ]:
today_share = {}
for t in worth_today:
    today_share[t] = worth_today[t] / my_worth

total_share = 0
for t in today_share:
    total_share += today_share[t]
print("I hold", len(today_share), "stocks today, and their shares add up to %.1f%%." % (100 * total_share))

#### The table

One row for each stock you hold. **Return** is how much its price moved since you bought it. **Dollars added** is
what that did to your fund.

In [ ]:
return_column = 'return since %s (%%)' % bought_day.strftime('%b %d')

rows = []
for t in bought_as:
    row = {}
    row['stock'] = t
    row['sector'] = record_stocks.get(t, '')
    row['share I bought (%)'] = round(100 * bought_as[t], 1)
    row['share today (%)'] = round(100 * today_share[t], 1)
    row[return_column] = round(100 * (prices.loc[last_close, t] / prices.loc[bought_day, t] - 1), 1)
    row['dollars added'] = round(dollars_added[t])
    rows.append(row)
by_stock = pd.DataFrame(rows).set_index('stock').sort_values('dollars added', ascending=False)
by_stock

In [ ]:
# the totals: the shares add up to 100%, and the dollars add up to my fund's gain or loss since I bought these stocks
total_added = 0
for t in dollars_added:
    total_added += dollars_added[t]

print("Share today, added up:   %.1f%%" % (100 * total_share))
print("Dollars added, added up: %+.0f" % total_added)
print("My fund since %s:      %+.0f   ($%.0f became $%.0f)" % (bought_day.strftime('%b %d'), my_worth - money_at_start, money_at_start, my_worth))

#### The same dollars as bars

Green added money. Black took it away.

In [ ]:
bars = by_stock['dollars added'].sort_values()
colors = []
for t in bars.index:
    if bars[t] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')

fig, panel = subplots(figsize=(10, max(4, 0.32 * len(bars))))
panel.barh(bars.index, bars.values, color=colors)
panel.axvline(0, color='gray', linewidth=1)
panel.set_title('Dollars each stock added (green) or took away (black)')
panel.set_xlabel('dollars')
show()

print("Helped the most:", by_stock.index[0], "(%+.0f dollars)" % by_stock['dollars added'].iloc[0])
print("Hurt the most:  ", by_stock.index[-1], "(%+.0f dollars)" % by_stock['dollars added'].iloc[-1])

#### By sector

The same table, added up by sector.

In [ ]:
by_sector = by_stock.groupby('sector')[['share today (%)', 'dollars added']].sum()
by_sector.sort_values('dollars added', ascending=False)

### 2c - My fund vs its no-rules twin

The **no-rules twin** is the plain Ms. Womack model from Studio 4 (mean-variance, the textbook version): the highest
average return for your risk limit, with no floor, no cap and no sector rule. It uses the same list of stocks and the
same risk limit as your fund. The only difference is your business rules, so the gap between the two is the difference
your rules made.

Both funds can lose money. The gap only says which one is ahead.

In [ ]:
twin_mix = versions[-1]['twin_mix']        # what the twin holds in my newest version

rows = {}
rows['my fund (with my rules)'] = {
    '$10,000 became': scorecard.loc[my_name, '$10,000 became'],
    'worst drop (%)': scorecard.loc[my_name, 'worst drop (%)'],
    'stocks held': len(bought_as),
    'biggest holding': '%s %.0f%%' % (pd.Series(bought_as).idxmax(), 100 * max(bought_as.values())),
}
rows['its no-rules twin'] = {
    '$10,000 became': scorecard.loc['its no-rules twin', '$10,000 became'],
    'worst drop (%)': scorecard.loc['its no-rules twin', 'worst drop (%)'],
    'stocks held': len(twin_mix),
    'biggest holding': '%s %.0f%%' % (pd.Series(twin_mix).idxmax(), 100 * max(twin_mix.values())),
}
pd.DataFrame(rows).T

In [ ]:
gap = scorecard.loc[my_name, '$10,000 became'] - scorecard.loc['its no-rules twin', '$10,000 became']

if gap >= 0:
    print("My fund is $%.0f AHEAD of its no-rules twin." % gap)
else:
    print("My fund is $%.0f BEHIND its no-rules twin." % (-gap))

**Read the table.** With no rules, Ms. Womack's model piles into a few stocks, so the twin rises and falls with
them. Your cap spreads the money out. Which one had the smaller worst drop?

### 2d - The days that mattered, and the news

A few days usually explain most of a month. Here are your fund's three best and three worst days, next to the S&P 500
on the same day.

**What moved it most** is the stock that pushed your fund the hardest that day: its share of your fund times its
return that day.

In [ ]:
my_days = 100 * my_line.pct_change().dropna()                # my fund's return each day, in %
spy_days = 100 * spy_line.pct_change().dropna()              # the S&P 500's
stock_days = 100 * prices.pct_change(fill_method=None)       # every stock's

best = my_days.sort_values(ascending=False).head(3)
worst = my_days.sort_values().head(3)

big_days = []
for day in list(best.index) + list(worst.index):
    # the version of my fund I held that day
    held = versions[0]['mix']
    for version in versions:
        if version['day'] < day:
            held = version['mix']

    # the stock that pushed my fund the hardest: its share times its return that day
    mover = None
    hardest_push = 0
    for t in held:
        push = held[t] * stock_days.loc[day, t]
        if abs(push) > abs(hardest_push):
            mover = t
            hardest_push = push

    big_days.append({'day': day, 'my fund (%)': round(my_days[day], 2), 'S&P 500 (%)': round(spy_days[day], 2),
                     'what moved it most': mover, 'its return that day (%)': round(stock_days.loc[day, mover], 1)})

big_table = pd.DataFrame(big_days).set_index('day')
big_table.index = big_table.index.strftime('%a %b %d')
big_table

#### What was in the news?

For each of those days, the next cell asks Google News for headlines about the stock that moved your fund the most,
from the day before to the day after. Read them: do they explain the move?

In [ ]:
import xml.etree.ElementTree as ET

# company names, so we can search for "Tesla stock" and not "TSLA stock"
companies = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv').set_index('ticker')['company']

for row in big_days:
    day = row['day']
    name = companies.get(row['what moved it most'], row['what moved it most'])

    # search from the day before to the day after
    after = (day - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    before = (day + pd.Timedelta(days=2)).strftime('%Y-%m-%d')
    search = '%s stock after:%s before:%s' % (name, after, before)
    link = 'https://news.google.com/rss/search?q=' + requests.utils.quote(search) + '&hl=en-US&gl=US&ceid=US:en'

    print("%s: my fund %+.2f%%. %s (%s) %+.1f%%" % (day.strftime('%a %b %d'), row['my fund (%)'], name,
                                                   row['what moved it most'], row['its return that day (%)']))
    try:
        feed = ET.fromstring(requests.get(link, timeout=20).content)
        shown = 0
        for item in feed.iter('item'):
            print("    -", item.findtext('title'))
            shown += 1
            if shown == 3:
                break
        if shown == 0:
            print("    (no headlines found)")
    except Exception as problem:
        print("    (Google News did not answer. Search for it yourself:", search + ")")
    print()

### Write it down (Studio 6 credit)

Double-click this cell and answer in three or four sentences, with numbers from the tables above.

**What happened to my fund since it started? Which two or three stocks explain most of it? Did my rules help?**

*My answer:*

# Part 3 - Look around

Three places to look before you decide.

### 3a - What the class holds

#### Who holds each stock?

Go through every fund in the class and write down, for each stock, the names of the funds that hold it.

In [ ]:
holders = {}          # stock -> the names of the funds that hold it
for fund in class_funds:
    for t in fund['weights']:
        if t not in holders:
            holders[t] = []
        holders[t].append(fund['name'])

print(len(class_funds), "student funds hold", len(holders), "different stocks.")
print("For example, XOM is held by:", holders.get('XOM', 'nobody'))

#### The class table

One row per stock: how many funds hold it, its return since the Nov 5 close (the day most of the class started, so
every stock is measured from the same day), and whether you hold it too.

In [ ]:
crowd_day = pd.Timestamp(crowd_file['crowd']['start'])      # the Nov 5 close
class_return = 'return since %s (%%)' % crowd_day.strftime('%b %d')

rows = []
for t in holders:
    row = {}
    row['stock'] = t
    row['funds holding it'] = len(holders[t])
    row[class_return] = round(100 * (prices.loc[last_close, t] / prices.loc[crowd_day, t] - 1), 1)
    if t in today_share:
        row['in my fund'] = 'yes'
    else:
        row['in my fund'] = ''
    row['held by'] = ', '.join(holders[t])
    rows.append(row)
class_table = pd.DataFrame(rows).set_index('stock')

print("The table has", len(class_table), "stocks. The first rows:")
class_table.head()

#### The most popular stocks

The twelve stocks the most funds hold. Green went up since Nov 5, red went down.

In [ ]:
popular = class_table.sort_values('funds holding it', ascending=False).head(12)

# color the return column: red for the most negative, green for the most positive
popular.style.background_gradient(cmap='RdYlGn', subset=[class_return], vmin=-15, vmax=15).format({class_return: '{:+.1f}'})

#### The class's biggest winners and losers

The eight best and eight worst stocks anyone in the class holds, since Nov 5, and **who holds them**. In class: who
has the winner? Ask them why they picked it.

In [ ]:
ranked = class_table.sort_values(class_return, ascending=False)
winners_and_losers = pd.concat([ranked.head(8), ranked.tail(8)])

winners_and_losers.style.background_gradient(cmap='RdYlGn', subset=[class_return], vmin=-15, vmax=15).format({class_return: '{:+.1f}'})

#### The whole class in one picture

Every fund is a row, with its biggest holding on the left. Each box is one stock, colored by its return since Nov 5.
A row of green is a fund having a good month.

In [ ]:
# one row per fund, one column per holding (biggest first); each cell is that stock's return since Nov 5
most_stocks = 0
for fund in class_funds:
    most_stocks = max(most_stocks, len(fund['weights']))

colors_grid = np.full((len(class_funds), most_stocks), np.nan)
labels_grid = []
fund_names = []
for i in range(len(class_funds)):
    fund_names.append(class_funds[i]['name'])
    mix = class_funds[i]['weights']
    labels = []
    k = 0
    for t in sorted(mix, key=mix.get, reverse=True):
        colors_grid[i, k] = class_table.loc[t, class_return]
        labels.append('%s\n%+.0f%%' % (t, class_table.loc[t, class_return]))
        k += 1
    labels_grid.append(labels)

fig, panel = subplots(figsize=(1.0 * most_stocks + 3, 0.62 * len(class_funds) + 1))
panel.imshow(colors_grid, cmap='RdYlGn', vmin=-15, vmax=15, aspect='auto')
for i in range(len(class_funds)):
    for k in range(len(labels_grid[i])):
        panel.text(k, i, labels_grid[i][k], ha='center', va='center', fontsize=7.5)
panel.set_yticks(np.arange(len(class_funds)))
panel.set_yticklabels(fund_names)
panel.set_xticks([])
panel.set_title('Every fund in the class: each stock and its return since %s' % crowd_day.strftime('%b %d'))
show()

#### What I am missing

The stocks your classmates hold that **you do not**: their five best returns and their five worst.

In [ ]:
not_mine = class_table[class_table['in my fund'] == ''].sort_values(class_return, ascending=False)
missing_out = pd.concat([not_mine.head(5), not_mine.tail(5)])

missing_out.style.background_gradient(cmap='RdYlGn', subset=[class_return], vmin=-15, vmax=15).format({class_return: '{:+.1f}'})

#### How different am I from the crowd?

For each stock, take the smaller of your share and the crowd's share, and add them up. That is the part of your fund
that is the same as the crowd fund. The rest is your own bet.

In [ ]:
same = 0
for t in today_share:
    same += min(today_share[t], crowd_mix.get(t, 0))

print("%.0f%% of my fund is the same as the class crowd fund." % (100 * same))
print("The other %.0f%% is my own bet." % (100 - 100 * same))

### 3b - The class crowd fund: does the class beat the market?

Suppose you did not pick stocks at all. On Nov 5 you put **the same number of dollars in every student's fund** and
then left it alone. That is the class crowd fund. With \$10,000 and 13 funds, each fund gets about \$769.

Does that beat the S&P 500?

In [ ]:
# both held from the same day: the Nov 5 close
crowd_since = hold(crowd_mix, crowd_day)
spy_since = hold({'SPY': 1.0}, crowd_day)

crowd_now = float(crowd_since.iloc[-1])
spy_now = float(spy_since.iloc[-1])

print("$10,000 in the class crowd fund since %s: $%.0f" % (crowd_day.strftime('%b %d'), crowd_now))
print("$10,000 in the S&P 500 since %s:          $%.0f" % (crowd_day.strftime('%b %d'), spy_now))
if crowd_now >= spy_now:
    print("The class is $%.0f AHEAD of the market." % (crowd_now - spy_now))
else:
    print("The class is $%.0f BEHIND the market." % (spy_now - crowd_now))

In [ ]:
fig, panel = subplots(figsize=(11, 4.5))
panel.plot(spy_since.index, spy_since.values, color='black', linewidth=2, label='S&P 500')
panel.plot(crowd_since.index, crowd_since.values, color='tab:purple', linewidth=3, label='class crowd fund')
panel.axhline(10000, color='lightgray', linewidth=1)
panel.set_title('$10,000 since the %s close: the class vs the market' % crowd_day.strftime('%b %d'))
panel.set_ylabel('dollars')
panel.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
panel.legend()
show()

#### Every fund in the crowd

Now each fund on its own, held from the same Nov 5 close. (This uses each fund's newest mix, so it can differ a little
from the leaderboard, where some funds started later.)

In [ ]:
rows = []
for fund in class_funds:
    line = hold(fund['weights'], crowd_day)
    rows.append({'fund': fund['name'], class_return: round(100 * (line.iloc[-1] / 10000 - 1), 2)})
fund_returns = pd.DataFrame(rows).set_index('fund').sort_values(class_return, ascending=False)

crowd_return = 100 * (crowd_now / 10000 - 1)
spy_return = 100 * (spy_now / 10000 - 1)

print("The average of the %d funds' returns: %+.2f%%" % (len(fund_returns), fund_returns[class_return].mean()))
print("The crowd fund's return:              %+.2f%%" % crowd_return)
print("They are the same number: owning an equal slice of every fund gives you the average fund.")
fund_returns

In [ ]:
bars = fund_returns[class_return].sort_values()
colors = []
for name in bars.index:
    if bars[name] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')

fig, panel = subplots(figsize=(10, 0.42 * len(bars) + 1.5))
panel.barh(bars.index, bars.values, color=colors)
panel.axvline(crowd_return, color='tab:purple', linewidth=2.5, label='class crowd fund')
panel.axvline(spy_return, color='black', linewidth=2, linestyle='--', label='S&P 500')
panel.set_xlabel(class_return)
panel.set_title('Every fund, the crowd and the market since %s' % crowd_day.strftime('%b %d'))
panel.legend()
show()

print("%d of %d funds beat the S&P 500." % ((bars > spy_return).sum(), len(bars)))
print("%d of %d funds beat the crowd." % ((bars > crowd_return).sum(), len(bars)))

**Read the picture.** The purple line always sits between the best bar and the worst bar, because the crowd's
return is the average of the funds' returns. Count the bars to the right of the purple line: one big winner pulls the
average above most of the funds.

### 3c - What the leaders hold

The leaderboard has a page called **Who holds what**: every fund in rank order with its stocks, colored the same way
(green up, red down since Nov 5). Click a fund's name there to see its own page.

**Who holds what:** https://storage.googleapis.com/opim5641-leaderboard-f26/holdings.html

**The leaderboard:** https://storage.googleapis.com/opim5641-leaderboard-f26/index.html

* What do the top three funds hold that you do not?
* Is one stock behind most of their lead, or is it spread out?
* Find the manager of the fund at the top and ask why they picked those stocks.

### 3d - My stocks this week, and the news

Each stock you hold: its return over the last five trading days and over the last month (about 21 trading days),
biggest movers of the week on top.

The cell after the table pulls this week's headlines for your three biggest movers. An earnings report? A new product?
Oil prices? Interest rates? Then look ahead: what is on the calendar between now and Dec 3 that could move your fund?

In [ ]:
rows = []
for t in today_share:
    row = {}
    row['stock'] = t
    row['share today (%)'] = round(100 * today_share[t], 1)
    row['last 5 days (%)'] = round(100 * (prices[t].iloc[-1] / prices[t].iloc[-6] - 1), 1)
    row['last month (%)'] = round(100 * (prices[t].iloc[-1] / prices[t].iloc[-22] - 1), 1)
    rows.append(row)
this_week = pd.DataFrame(rows).set_index('stock')

# biggest movers first, up or down
order = this_week['last 5 days (%)'].abs().sort_values(ascending=False).index
this_week = this_week.loc[order]
this_week

#### This week's headlines for my three biggest movers

In [ ]:
week_ago = (last_close - pd.Timedelta(days=7)).strftime('%Y-%m-%d')

for t in this_week.index[:3]:
    name = companies.get(t, t)
    search = '%s stock after:%s' % (name, week_ago)
    link = 'https://news.google.com/rss/search?q=' + requests.utils.quote(search) + '&hl=en-US&gl=US&ceid=US:en'

    print("%s (%s): %+.1f%% in the last 5 days" % (name, t, this_week.loc[t, 'last 5 days (%)']))
    try:
        feed = ET.fromstring(requests.get(link, timeout=20).content)
        shown = 0
        for item in feed.iter('item'):
            print("    -", item.findtext('title'))
            shown += 1
            if shown == 4:
                break
    except Exception as problem:
        print("    (Google News did not answer. Search for it yourself:", search + ")")
    print()

### Write it down

Double-click and add two notes. One line each is fine.

**One thing I learned from another fund:**

**One thing in the news that matters to my fund:**

# Part 4 - What does the model say today?

In Studio 5 the model read five years of monthly returns, October 2021 to September 2026. Two things have changed
since then:

1. **A new month has closed.** The model can read October now.
2. **Your fund has drifted.** The stocks that went up are a bigger share of your money than the model gave them, and
   the ones that fell are smaller.

So we solve the same model again and compare three versions of your fund:

| | What it is |
|---|---|
| **when I bought** | the shares the model gave you when you sent the fund |
| **today** | what those shares have drifted to (Part 2b) |
| **the model says now** | the same model, solved again with the newest months |

Trading back to the shares the model wants is called **rebalancing**.

### Step 1 - The model

This is `solve_portfolio` from Studio 5, Step 5, copied over without changes. Run it.

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, \sigma_{ij} \, x_j \le \text{risk limit} && \text{(Ms. Womack's risk)} \\
& \text{floor} \cdot y_k \le x_k \le \text{cap} \cdot y_k, \quad y_k \in \{0, 1\} && \text{(your business rules)}
\end{aligned}
$$

In [ ]:
def solve_portfolio(risk_limit, use_rules):
    model = ConcreteModel()

    # how much of the money goes in each stock
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # the risk: the weighted covariance matrix, summed
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        # the yes/no decision: do we hold stock s?
        model.y = Var(stocks, domain=Binary)
        for s in stocks:
            model.y[s] = 1
        for s in stocks:
            if rules['max_share'] is not None:
                model.constraints.add(model.x[s] <= rules['max_share'] * model.y[s])
            else:
                model.constraints.add(model.x[s] <= model.y[s])
            if rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= rules['min_share'] * model.y[s])
        for sector in rules['must_have_sectors']:
            picks = 0
            count = 0
            for s in stocks:
                if my_stocks.get(s) == sector:
                    picks += model.y[s]
                    count += 1
            if count > 0:    # a sector you have no stocks in is skipped
                model.constraints.add(picks >= 1)
        if rules['max_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held <= rules['max_stocks'])
        # the extra linking rules
        if extra_rules['sector_cap'] is not None:
            for sector in set(my_stocks.values()):
                in_sector = 0
                count = 0
                for s in stocks:
                    if my_stocks.get(s) == sector:
                        in_sector += model.x[s]
                        count += 1
                if count > 0:
                    model.constraints.add(in_sector <= extra_rules['sector_cap'])
        if extra_rules['min_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held >= extra_rules['min_stocks'])
        for s in extra_rules['must_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 1)
        for s in extra_rules['never_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 0)
        for pair in extra_rules['either_or']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] + model.y[pair[1]] <= 1)
        for pair in extra_rules['if_then']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] <= model.y[pair[1]])
        results = SolverFactory('bonmin', executable=bonmin_path).solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights, 'risk_limit': risk_limit}


print("solve_portfolio is ready")

### Step 2 - A stock in the news: would the model sell NVIDIA?

NVIDIA has been all over the news this month. Would Ms. Womack's model sell it? That depends on **how much history
the model reads**.

#### NVIDIA lately

In [ ]:
nvda = prices['NVDA']

# month-end prices, complete months only (this month is not over yet)
month_end = prices.groupby(prices.index.to_period('M')).last()
month_end = month_end[month_end.index < last_close.to_period('M')]

print("NVIDIA since the Nov 5 close:  %+.1f%%" % (100 * (nvda.iloc[-1] / nvda.loc['2026-11-05'] - 1)))
print("NVIDIA this month so far:      %+.1f%%" % (100 * (nvda.iloc[-1] / month_end['NVDA'].iloc[-1] - 1)))
print()
print("Its last six complete months (%):")
(100 * month_end['NVDA'].pct_change().tail(6)).round(1).to_frame('NVDA monthly return (%)')

#### An example fund

To see what the model does with NVIDIA we need a fund that could hold it: the 20-stock starter list from Studio 5,
with the Studio 5 starting rules (at least 2%, at most 10%, one stock each from Technology, Health Care and Energy).

In [ ]:
my_stocks = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'NVDA': 'Technology',
    'GOOGL': 'Communication Services', 'META': 'Communication Services',
    'JPM': 'Financials', 'GS': 'Financials', 'V': 'Financials',
    'JNJ': 'Health Care', 'LLY': 'Health Care', 'UNH': 'Health Care',
    'XOM': 'Energy', 'CVX': 'Energy',
    'KO': 'Consumer Staples', 'WMT': 'Consumer Staples', 'COST': 'Consumer Staples',
    'AMZN': 'Consumer Discretionary', 'HD': 'Consumer Discretionary',
    'CAT': 'Industrials', 'LMT': 'Industrials',
}
rules = {'min_share': 0.02, 'max_share': 0.10, 'must_have_sectors': ['Technology', 'Health Care', 'Energy'], 'max_stocks': None}
extra_rules = {'sector_cap': None, 'min_stocks': None, 'must_hold': [], 'never_hold': [], 'either_or': [], 'if_then': []}

print(len(my_stocks), "stocks in the example fund")

#### Solve it three times: 5 years, 3 years and 1 year of history

Everything else stays the same. For each history the loop works out NVIDIA's average month, where that ranks among
the 20 stocks, and how much of the fund the model gives it. The risk limit is the even split's risk, as in Studio 5.

In [ ]:
rows = []
for years in [5, 3, 1]:
    # the model's two ingredients, from the last `years` years of monthly returns
    returns = month_end[list(my_stocks)].pct_change(fill_method=None).iloc[1:].tail(12 * years)
    stocks = list(returns.columns)
    df_return = returns.mean()
    df_cov = returns.cov()

    # the risk limit: the risk of an even split (Studio 5, Step 6)
    even_risk = 0
    for i in stocks:
        for j in stocks:
            even_risk += df_cov.loc[i, j] / len(stocks) ** 2

    with_rules = solve_portfolio(even_risk, use_rules=True)
    no_rules = solve_portfolio(even_risk, use_rules=False)

    row = {}
    row['history'] = '%d year(s)' % years
    row['NVDA average month (%)'] = round(100 * df_return['NVDA'], 2)
    row['its rank of 20'] = int(df_return.rank(ascending=False)['NVDA'])
    row['NVDA share, with rules (%)'] = round(100 * with_rules['weights']['NVDA'], 1)
    row['NVDA share, no rules (%)'] = round(100 * no_rules['weights']['NVDA'], 1)
    rows.append(row)

pd.DataFrame(rows).set_index('history')

**Read the table.** With five or three years of history, NVIDIA still has one of the best average months on the
list, so the model keeps buying as much as the cap allows. One bad month barely moves a five-year average, and the
model has **not seen November at all**, because the month is not over. With one year of history the picture changes.

A short history reacts faster. A long history is steadier. You choose with `fit_years` in the next step.

### Step 3 - The scenario

Now your fund. This cell sets up the model's inputs. Run it **as it is** the first time: same stocks, same rules, same risk limit,
five years of history.

Later, if you want to change something, come back here, **remove the `#`** from a line, edit it, and run Part 4 again
from this cell down.

In [ ]:
# start from my fund exactly as it is on the leaderboard (so running this cell twice never changes things twice)
my_stocks = dict(record_stocks)
rules = dict(record_rules)
extra_rules = dict(record_extra)
my_risk_limit = record_risk_limit
fit_years = 5                                   # how many years of monthly returns the model reads

# ---- MY CHANGES: remove the # from a line to use it, and edit the number or the ticker ----
# rules['max_share'] = 0.15                     # at most 15% in one stock
# rules['min_share'] = 0.03                     # at least 3% in any stock I hold
# rules['max_stocks'] = 8                       # hold no more than 8 stocks
# rules['must_have_sectors'] = ['Technology']   # at least one stock from each sector listed
# extra_rules['never_hold'] = ['TSLA']          # the model may not buy these
# extra_rules['must_hold'] = ['WMT']            # the model must hold these
# my_stocks['NVDA'] = 'Technology'              # add a stock to my list (ticker and its sector)
# del my_stocks['PNC']                          # take a stock off my list
# fit_years = 1                                 # read the last 1 year instead of 5 (see the NVIDIA example)
# my_risk_limit = record_risk_limit * 0.8       # 20% less risk than I allowed in Studio 5

print("Stocks on my list:", len(my_stocks))
print("Rules:            ", rules)
print("Extra rules:      ", extra_rules)
print("Risk limit:        %.4f" % my_risk_limit)
print("History:          ", fit_years, "years")

### Step 4 - The newest monthly returns

Same steps as Studio 5: month-end prices, then returns, then drop any stock that is missing months.

#### Month-end prices

The last closing price of each month. This month is not over yet, so it is left out.

In [ ]:
# a stock I just added to my list in the scenario cell: download its prices first
for t in my_stocks:
    if t not in prices.columns:
        new_prices = yf.download(t, start='2021-09-01', interval='1d', auto_adjust=True, progress=False)['Close'].squeeze()
        prices[t] = new_prices.reindex(prices.index).ffill()
        print("downloaded", t)

# the last price of each month
month_end = prices.groupby(prices.index.to_period('M')).last()

# complete months only
month_end = month_end[month_end.index < last_close.to_period('M')]

print("Month-end prices from", month_end.index[0], "to", month_end.index[-1])

#### Monthly returns

$$r_t = \frac{P_t}{P_{t-1}} - 1$$

In [ ]:
# monthly returns for the stocks on my list
returns = month_end[list(my_stocks)].pct_change(fill_method=None)

# keep the last fit_years years
returns = returns.iloc[1:].tail(12 * fit_years)

# drop any stock that is missing months (a company that did not exist yet)
complete = []
for t in returns.columns:
    if returns[t].isna().sum() == 0:
        complete.append(t)
    else:
        print("dropped", t, "(missing months)")
returns = returns[complete]

print("The model reads", len(returns), "months,", returns.index[0], "to", returns.index[-1], "for", len(returns.columns), "stocks.")
returns.tail(3).round(3)

#### The model's two ingredients

Each stock's average month, and the covariance matrix: the same `df_return` and `df_cov` as Studio 5.

In [ ]:
stocks = list(returns.columns)
df_return = returns.mean()
df_cov = returns.cov()

print("Best average month: ", df_return.idxmax(), "%.2f%%" % (100 * df_return.max()))
print("Worst average month:", df_return.idxmin(), "%.2f%%" % (100 * df_return.min()))

### Step 5 - Solve it

Once with your rules (your fund) and once without (its twin), at your risk limit.

In [ ]:
target = solve_portfolio(my_risk_limit, use_rules=True)
twin_target = solve_portfolio(my_risk_limit, use_rules=False)

if target is None or twin_target is None:
    print("No portfolio keeps all of the rules at this risk limit.")
    print("Go back to the scenario cell: loosen a rule, add stocks, or allow more risk. Then run Part 4 again.")
else:
    print("Solved.")
    print("The model's average month: %.2f%%" % (100 * target['return']))
    print("Its risk: %.4f (my limit is %.4f)" % (target['risk'], my_risk_limit))

The model gives every stock a number, and most of them are zero. Keep the stocks it actually holds (more than
0.1% of the money).

In [ ]:
model_mix = {}
for t in target['weights']:
    if target['weights'][t] > 0.001:
        model_mix[t] = round(target['weights'][t], 6)

twin_model_mix = {}
for t in twin_target['weights']:
    if twin_target['weights'][t] > 0.001:
        twin_model_mix[t] = round(twin_target['weights'][t], 6)

print("The model holds %d stocks today." % len(model_mix))
print("Its no-rules twin holds %d." % len(twin_model_mix))

### Step 6 - The trades

The three versions of your fund side by side. **Trade (\$)** is what you would buy (+) or sell (-) to move from what
you hold today to the model's answer:

$$\text{trade} = \text{my fund's dollars} \times (\text{the model's share} - \text{my share today})$$

In [ ]:
every_stock = sorted(set(list(bought_as) + list(today_share) + list(model_mix)))

rows = []
for t in every_stock:
    row = {}
    row['stock'] = t
    row['when I bought (%)'] = round(100 * bought_as.get(t, 0), 1)
    row['today (%)'] = round(100 * today_share.get(t, 0), 1)
    row['the model says now (%)'] = round(100 * model_mix.get(t, 0), 1)
    row['trade ($)'] = round(my_worth * (model_mix.get(t, 0) - today_share.get(t, 0)))
    rows.append(row)
trades = pd.DataFrame(rows).set_index('stock').sort_values('trade ($)', ascending=False)
trades

In [ ]:
# every dollar I buy is a dollar I sell, so the money that moves is half of the total
dollars_traded = trades['trade ($)'].abs().sum() / 2

print("To move to the model's answer I would trade $%.0f." % dollars_traded)
print("That is %.0f%% of my fund." % (100 * dollars_traded / my_worth))

#### The picture

For each stock, two bars: the share of your fund it is **today** (black), and the share **the model says it should be
now** (green). Where the green bar is longer, the model wants you to buy. Where the black bar is longer, it wants you
to sell.

In [ ]:
bars = trades.sort_values('the model says now (%)')
spots = np.arange(len(bars))

fig, panel = subplots(figsize=(10, max(4, 0.36 * len(bars))))
panel.barh(spots + 0.2, bars['today (%)'], height=0.4, color='black', label='my share today')
panel.barh(spots - 0.2, bars['the model says now (%)'], height=0.4, color='tab:green', label='the share the model says now')
panel.set_yticks(spots)
panel.set_yticklabels(bars.index)
panel.set_xlabel('share of my fund (%)')
panel.set_title('Each stock: my share today vs the share the model says now')
panel.legend()
show()

**Read the trades.** A short list means your fund is still close to what the model wants. A long list means
your fund has drifted a long way, or the new month changed the model's answer.

**Want to try something?** Go back to the scenario cell in Step 3, remove a `#`, and run Part 4 again. Try a looser
cap, a shorter history, or a stock you saw in a classmate's fund. Each run takes a few seconds.

# Part 5 - My decision

Two choices.

| `choice` | What it does |
|---|---|
| `'hold'` | no trades: my fund stays exactly as it is today |
| `'rebalance'` | trade to the shares the model says now (Part 4). This trims any stock that has grown past my cap, follows the newest months, and uses any change I made in the scenario cell |

Then write your **reason** in your own words, one or two sentences. Use what you found tonight: a number from Part 2,
something another fund did, something in the news, the size of the trade list.

In [ ]:
decision = {
    'choice': 'hold',       # 'hold' or 'rebalance'
    'reason': '',           # one or two sentences, in my own words
}

The next cell turns your choice into the fund you will hold. Change your choice and run it again as often as
you like: nothing is sent yet.

In [ ]:
if decision['choice'] == 'hold':
    # no trades: tomorrow's fund is today's fund
    new_mix = {}
    for t in today_share:
        new_mix[t] = round(today_share[t], 6)
    # the twin holds too. (If I have re-sent my fund before, the twin goes back to its newest mix.)
    new_twin_mix = twin_mix

elif decision['choice'] == 'rebalance':
    # the shares the model says now, from Part 4
    new_mix = model_mix
    new_twin_mix = twin_model_mix

else:
    print("decision['choice'] should be 'hold' or 'rebalance'.")

print("My decision:", decision['choice'])
print("My fund will hold", len(new_mix), "stocks:")
for t in sorted(new_mix, key=new_mix.get, reverse=True):
    print("   %-6s %5.1f%%" % (t, 100 * new_mix[t]))

### Send it to the leaderboard

Fill in your NetID and the class code (Dr. Dave writes it on the board), then run the next two cells.

Your decision is saved with today's date and your reason. It counts from the next market close, and your fund keeps
its value: the new version starts with whatever the fund is worth at that close.

If you hold, send it anyway: your reason is saved in your fund's history, and you will use it in your report.

In [ ]:
netid = ""            # my UConn NetID, for example abc12345
class_code = ""       # Dr. Dave writes it on the board

#### What the leaderboard will save

In [ ]:
entry = {
    'netid': netid.strip().lower(),
    'nickname': my_name,
    'theme': my_record['theme'],
    'reason': ("Studio 6 (%s): %s" % (decision['choice'], decision['reason'].strip()))[:200],
    'class_code': class_code.strip(),
    'portfolios': {'no_rules': new_twin_mix, 'with_rules': new_mix},
    'risk_limit': float(my_risk_limit),
    'rules': rules,
    'extra_rules': extra_rules,
    'stock_theme': newest.get('stock_theme'),
    'sectors': my_stocks,
    'data_window': [str(returns.index[0]), str(returns.index[-1])],
    'decision': decision['choice'],
    'notebook': 'studio6',
}

# is anything still missing?
missing = []
if entry['netid'] == '':
    missing.append('netid')
if entry['class_code'] == '':
    missing.append('class_code')
if len(decision['reason'].strip()) < 30:
    missing.append("decision['reason'] (write one or two full sentences)")
print("Still missing:", missing)

#### Send

Colab asks you to sign in with Google (a pop-up: pick your account and click Allow). Then the leaderboard answers
with a receipt.

In [ ]:
if 'https://storage.googleapis.com/opim5641-leaderboard-f26/records/8000437f223b5f668e7dee73.json' in record_link:
    print("Not sent: this is Dr. Dave's example fund. Paste YOUR record link in Part 1, Run all, then come back.")
elif len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "and run the last two cells again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and send.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post('https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio',
                           data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        print("Saved. This is what the leaderboard has:")
        print("  fund:        ", receipt['nickname'])
        print("  NetID:       ", receipt['netid'], "(signed in as " + receipt['google_email'] + ")")
        print("  reason:      ", receipt['reason'])
        print("  stocks held: ", receipt['holdings']['with_rules'])
        print("  It counts from the next market close (4 PM Eastern on a weekday).")
        print("  My record link stays the same:", receipt['record'])
    else:
        print("Not accepted:", receipt.get('problem'))

### Before you leave tonight (Studio 6 credit)

Double-click and answer. Two or three sentences each, with numbers.

**1. What did the model say today, and how different is it from what I hold?**

*My answer:*

**2. What did I decide, and what convinced me?**

*My answer:*

Then save this notebook (File, Download, Download .ipynb) and hand it in on HuskyCT.

# After tonight

You can change your fund until the market closes on **Thursday, Dec 3**.

1. Open this notebook, then Runtime, Run all. Parts 2 to 4 use the newest prices.
2. Set your decision in Part 5 and send it.

Each send counts from the next market close and is saved with its date and your reason.

# The final project (individual, due Monday, Dec 7, 11:59 PM)

1. **After the Dec 3 close** (Thursday, 4 PM), open this notebook and **Run all**. The race is over at that close, so
   the numbers will not change while you write.
2. Use the numbers printed below in a **2-page report** (a PDF on HuskyCT), in five parts.
3. Add the picture from the last cell as **page 3**.

| Part | Length | What goes in it |
|---|---|---|
| 1 - My fund and my rules | ¼ page | your stocks, your rules, and why you chose them in Studio 5 |
| 2 - What happened before Studio 6 | ½ page | the race up to Nov 18: the stocks that helped and hurt, what your rules did, how you compared with the crowd and the S&P 500 |
| 3 - My decision on Nov 18 | ½ page | what the model said, the size of the trades, what you decided and why |
| 4 - What happened after | ½ page | Nov 19 to Dec 3: every later change and its reason, and your changes vs never touching the fund |
| 5 - What I learned | ¼ page | what you would do differently, and what you would hold on Jan 1, 2027 |

### The numbers for my report

#### The scorecard

In [ ]:
print("Through the close of", last_close.strftime('%a %b %d, %Y'))
print("My fund: ", my_name)
print("My rules:", record_rules)
scorecard

#### Every version of my fund, and what happened after it started

In [ ]:
rows = []
for k in range(len(versions)):
    day = versions[k]['day']
    row = {}
    row['version'] = k + 1
    row['counts from'] = day.strftime('%a %b %d')
    row['my fund since (%)'] = round(100 * (my_line.iloc[-1] / my_line.loc[day] - 1), 2)
    row['S&P 500 since (%)'] = round(100 * (spy_line.iloc[-1] / spy_line.loc[day] - 1), 2)
    row['my reason'] = versions[k]['reason']
    rows.append(row)
pd.DataFrame(rows).set_index('version')

#### My changes vs never touching the fund

Take your **first** version and hold it the whole time, with no changes. Then compare it with what your fund is
actually worth.

In [ ]:
untouched_line = hold(versions[0]['mix'], first_day)
never_touched = float(untouched_line.iloc[-1])

if len(versions) == 1:
    print("I have one version of my fund, so there are no changes to compare yet.")
else:
    print("My first version, never changed: $%.0f" % never_touched)
    print("My fund, with my changes:        $%.0f" % my_worth)
    if my_worth >= never_touched:
        print("My changes put me $%.0f AHEAD." % (my_worth - never_touched))
    else:
        print("My changes put me $%.0f BEHIND." % (never_touched - my_worth))

### My fund in one picture (page 3 of the report)

The race with each of your changes marked, and the stocks that drove it. The cell saves `my_fund_picture.png`, and in
Colab it downloads automatically.

In [ ]:
fig, (left, right) = subplots(1, 2, figsize=(15, 6))
fig.suptitle(my_name + ', through ' + last_close.strftime('%b %d, %Y'), fontsize=15, fontweight='bold')

# left: the race
left.plot(spy_line.index, spy_line.values, color='black', linewidth=2, label='S&P 500')
left.plot(untouched_line.index, untouched_line.values, color='gray', linewidth=1.5, linestyle='--', label='if I had never changed it')
left.plot(my_line.index, my_line.values, color='tab:green', linewidth=3, label='my fund')
for k in range(1, len(versions)):
    left.axvline(versions[k]['day'], color='tab:green', linestyle=':', linewidth=1)
left.axhline(10000, color='lightgray', linewidth=1)
left.set_title('$10,000 since %s (dotted lines: my changes)' % first_day.strftime('%b %d'))
left.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))
left.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
left.legend()

# right: the ten stocks that mattered most
biggest = by_stock['dollars added'].abs().sort_values(ascending=False).head(10).index
bars = by_stock.loc[biggest, 'dollars added'].sort_values()
colors = []
for t in bars.index:
    if bars[t] >= 0:
        colors.append('tab:green')
    else:
        colors.append('black')
right.barh(bars.index, bars.values, color=colors)
right.axvline(0, color='gray', linewidth=1)
right.set_title('Dollars each stock added or took away')

fig.savefig('my_fund_picture.png', dpi=150)
show()

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('my_fund_picture.png')